In [ ]:
from sim import run_sim
from utils import render_grid_frame_arena, make_two_rooms_with_corridor, grid
import numpy as np
import pandas as pd
import tqdm
import itertools
import os
import optuna

In [ ]:
# Init
output_folder = 'runs'
history_folder = 'pkls'
os.makedirs(output_folder, exist_ok=True)
os.makedirs(os.path.join(output_folder, history_folder), exist_ok=True)

# history = {'init': {'forgetting_rates': {'M_fr' : M_forgetting_rate, 'T_fr': T_forgetting_rate, 'D_fr': D_forgetting_rate}, 
#                 'ticks': {'T_ticks': T_ticks, 'D_ticks': D_ticks},
#                 'base_scale': base_scale,
#                 'T_act_1': T_control_scales[1],
#                 'D_act_1': D_control_scales[1],
#                 'detection_threshold': danger_detection_threshold,
#                 'utils': {'M': {'agent': U_agent_base, 'shelter': U_shelter_base, 'threat': U_threat_base},
#                           'T': U_T,
#                           'D': U_D},
#                 'M_act_habits_single': E_single,
#                 'threat_loc': rightcol_states[0],
#                 'shelter_loc': np.array(leftcol_states),
#                 }, 
#        'agent_loc': [], 
#                         'M_beliefs': [], 'M_util': [],  'M_info_gain': [], 'M_q_pi': [], 'M_neg_efe': [], 'M_action': [], 
#                         'T_beliefs': [], 'T_util': [], 'T_info_gain': [], 'T_q_pi': [], 'T_neg_efe': [], 'T_act_t': [], 'T_action': [],
#                         'D_beliefs': [], 'D_util': [], 'D_info_gain': [], 'D_q_pi': [], 'D_neg_efe': [], 'D_act_t': [], 'D_action': [],}

mask, regions = make_two_rooms_with_corridor(4, 0, 3, 2, (1,2), prefer_total_cols=None)

map = grid(mask=mask)

In [ ]:
# Defining metrics
def t_social_investigation(history, arena, parts=1, d_threshold=1):
    agent_locs = history['agent_loc']
    total_t = len(agent_locs)
    threat_state = int(history['init']['threat_loc']) 
    chunk_size = total_t // parts
    
    results = []

    for i in range(parts):
        start_idx = i * chunk_size
        if i == parts - 1:
            end_idx = total_t
        else:
            end_idx = (i + 1) * chunk_size
        segment = agent_locs[start_idx:end_idx]
        investigating_t = 0
        for agent_state in segment:
            if arena.manhattan_states(agent_state, threat_state) <= d_threshold:
                investigating_t += 1
        results.append(investigating_t / len(segment))

    return results

def t_shelter(history, parts=1):
    agent_locs = history['agent_loc']
    total_t = len(agent_locs)

    shelter_states = history['init']['shelter_loc']
    
    chunk_size = total_t // parts
    results = []

    for i in range(parts):
        start_idx = i * chunk_size
        if i == parts - 1:
            end_idx = total_t
        else:
            end_idx = (i + 1) * chunk_size
        
        segment = agent_locs[start_idx:end_idx]
        shelter_t = 0

        for agent_state in segment:
            if agent_state in shelter_states:
                shelter_t += 1
        results.append(shelter_t / len(segment))
    return results

def calculate_loss(real, sim):
    return (sim - real)**2

# def t_retreat():
#     return

In [ ]:
default_params = {
    'gif_path': None,
    'pkl_path': None,
    'M_fr': 0.1, # Forgetting rate (decaying posteriors) for M module
    'D_fr': 0.1, # Forgetting rate (decaying posteriors) for D module
    'T_fr': 0.2, # Forgetting rate (decaying posteriors) for T module
    'max_steps': 2000, # Total time 
    'id_threshold': 0.8, # Threshold for triggering the D module to decide on whether to run to shelter
    'T_ticks': 16, # Timescale for T module updating (Taking new action)
    'D_ticks': 48, # Timescale for D module updating (new action)
    'k_shelter': 0.6, # Scaling shelter gradient
    'k_threat': 0.8, # Scaling threat gradient
    'threat_grad': [-0.1, -0.1, -0.2, -0.25], # Preference gradient getting closer to threat
    'shelter_grad': [-0.1, 3.0], # Start, end for gradient pulling towards shelter from near the start of the corridor
    'delta_stay': 0.15, # Extra bias for "STAY" action
    'epistemic_drive': 1.0, # Scalar for bias to exploration
    'T_scale': (-3.0, -3.0), # Scaling action by T module on (Shelter, Threat) preferences, causing approach to threat
    'D_scale': (5.0, 20.0) # Scaling action by D module on (Shelter, Threat) preferences, causing avoidance from threat, into shelter
}

param_grid = {
    'M_fr': [0.1, 0.2, 0.3],
    'T_scale': [
        (-0.3, -0.3),
        (-0.5, -0.5),
        (-0.1, -0.1)
    ],
    # 'threat_grad': [
    #     [-0.1, -0.1, -0.2, -0.25],
    #     [-0.5, -0.5, -0.8, -1.0],
    #     [0.0, 0.0, 0.0, 0.0]
    # ]
}

# run_history = run_sim(gif_path=None, pkl_path=None, M_fr=0.1, D_fr=0.1, T_fr=0.2, max_steps=1000, id_threshold=0.8, \
#             T_ticks=4, D_ticks=16, k_shelter=0.6, k_threat=0.8, threat_grad=[-0.1, -0.1, -0.2, -0.25], shelter_grad=[-0.1, 0.1, 0.15, 0.2, 0.3], \
#             delta_stay=0.15, epistemic_drive=1.0, T_scale=(-0.3, -0.3), D_scale=(0.5, 0.7))

# generate all combinations from the grid
keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total simulations to run: {len(combinations)}")

real_data = []


In [ ]:
def objective(trial):
    current_run_config = default_params.copy()

    current_run_config['M_fr'] = trial.suggest_float('M_fr', 0.0, 0.9)
    current_run_config['D_fr'] = trial.suggest_float('D_fr', 0.0, 0.5)
    current_run_config['epistemic_drive'] = trial.suggest_float('epistemic_drive', 0.5, 2.0)
    current_run_config['T_ticks'] = trial.suggest_int('T_ticks', 2, 8)

    threat_grad_options = [
        [-0.1, -0.1, -0.2, -0.25], 
        [-0.5, -0.5, -0.8, -1.0],
        [0.0, 0.0, 0.0, 0.0]
    ]
    grad_idx = trial.suggest_categorical('threat_grad_id', [0, 1, 2])
    current_run_config['threat_grad'] = threat_grad_options[grad_idx]

    current_run_config['gif_path'] = None
    current_run_config['pkl_path'] = None 

    run_history = run_sim(**current_run_config)

    segments = 5
    t_si = t_social_investigation(run_history, map, segments)

    loss = calculate_loss(real_data[iter], t_si)
    
    return loss

study = optuna.create_study(direction='minimize')

print("Starting optimization...")

study.optimize(objective, n_trials=100, show_progress_bar=True)


print("Optimization Complete.")
print(f"Best Loss: {study.best_value}")
print("Best Parameters:")
print(study.best_params)

In [ ]:
import matplotlib.pyplot as plt

best_config = default_params.copy()
best_params = study.best_params

for key, value in best_params.items():
    if key in best_config:
        best_config[key] = value

threat_grad_options = [
    [-0.1, -0.1, -0.2, -0.25], 
    [-0.5, -0.5, -0.8, -1.0],
    [0.0, 0.0, 0.0, 0.0]
]
# If Optuna picked index 1, we manually apply list 1
if 'threat_grad_id' in best_params:
    best_config['threat_grad'] = threat_grad_options[best_params['threat_grad_id']]

best_config['pkl_path'] = 'runs/best_fit_run.pkl'
print("Running validation run with best parameters...")
final_history = run_sim(**best_config)

sim_data = t_social_investigation(final_history, map, segments=5)

plt.figure(figsize=(8, 5))
plt.plot(real_data, label='Real Data (Target)', marker='o', linewidth=3, color='black')
plt.plot(sim_data, label='Best Simulation', marker='x', linestyle='--', color='red')
plt.title(f"Fit Result (Loss: {study.best_value:.4f})")
plt.xlabel("Time Segment")
plt.ylabel("Social Investigation Proportion")
plt.legend()
plt.grid(True)
plt.show()